# Phase 0.7 round 2 — fine-tune SmolVLA on D1, the varied dataset

One of **two notebooks**, one per policy. Round 2 trains only SmolVLA; π0.5 is gated on the
frozen-feature probe in [§12.7](Details/phase07_portable_rig.md) and is not run here.

**This is a controlled comparison against round 1, not a fresh start.** Everything is held identical to
[the round-1 SmolVLA notebook](colab_phase07_smolvla.ipynb) — recipe,
batch, seed, holdout mechanism, eval cadence — and **only the dataset changes**. D1 is the same size as
D0 (50 episodes), so the comparison is size-matched and the single variable is the data's character.
That is what [§12.1](Details/phase07_portable_rig.md) pre-registered.

| | |
|---|---|
| Dataset | `so101_tray_cylinder_varied_20261013_082911_trimmed` — 50 episodes, **head-trimmed, 2 s tail kept** |
| Trained from | **`lerobot/smolvla_base`, fresh** — not warm-started from the round-1 checkpoint |
| Steps | **30,000** — 30k extends past round 1's 20k because round 1 found the *later* checkpoint better on the robot despite a worse loss — **the 20k checkpoint is still written**, so the matched comparison and the train-longer arm both come out of one run |
| GPU | **Colab L4** — batch 64 peaked at 16.5 of 24 GiB in round 1, which does not fit the 5080's 15.5 GiB, and lerobot has no gradient accumulation to work around it |
| Expected runtime | **~21 h** for 30k steps on an L4 — Part B prints the real figure |
| Output | `HALDijkstraaa/smolvla_tray_cylinder_s1000_d1` (private), checkpointed to the Hub as it goes |
| Sibling | [ACT](colab_phase07_r2_act.ipynb) |

**Why fresh rather than continued.** Warm-starting from the round-1 checkpoint would carry D0's
low-divergence structure into the run and confound the test; it would also make prediction **P3**
(ACT's KLD should no longer collapse) unmeasurable, because KLD behaviour on a warm start is not
comparable to KLD behaviour from scratch.

**What D1 changed, measured.** Reach-phase divergence between demonstrations rose **0.322 → 0.478**
(+48%, p < 0.0001; +0.113 and still p < 0.0001 after matching on position spread), the idle tail went
**45 → 95 frames**, and the 9-block position structure is **gone** (27× separation → 1×). The
pre-registered predictions that follow from it are in
[§12.1](Details/phase07_portable_rig.md) — read them before reading the results.


---
# Part A — Setup

Written to run on a **bare machine** — CUDA drivers and Python only — as well as on Colab. Run the cells
in order: they set up paths, then credentials, then install, and only then import anything that needs
torch.

## 1 — Where everything goes

### Why this comes first

Run this **before the install**, because it decides where ~7 GB of
model weights, dataset and checkpoints land.

Hosts differ: Colab has one big disk at `/content`, while a rented GPU box typically has a **small system
disk** (often 30 GB, holding `/root` and therefore `~/.cache`) plus a **large data disk** mounted
elsewhere. The Hugging Face cache defaults to `~/.cache/huggingface`, so on such a host the base model
would silently fill the system disk and the run would die mid-download.

This cell picks the biggest sensible working directory, points `HF_HOME` and `HF_LEROBOT_HOME` at it, and
prints the layout. Override with `PHASE07_WORK=/path` if it guesses wrong.

In [ ]:
import os, shutil
from pathlib import Path

CANDIDATES = ["/root/gpufree-data", "/root/autodl-tmp", "/workspace", "/content", "/mnt/data", "/data"]


def _du(path):
    s = shutil.disk_usage(path)
    return s.total / 1e9, s.free / 1e9


# FIRST MATCH in the list above wins, deliberately - the list is ordered by how likely a path is to
# be storage allocated TO YOU. Choosing by "most free space" instead would be wrong on this kind of
# host: it has a 98 GB allocated data disk at /root/gpufree-data and also a shared 1 TB /data that
# is 81% full, and the shared one shows more free space while being the wrong place to write 85 GB.
# All candidates are printed so a bad guess is visible, and PHASE07_WORK overrides.
if os.environ.get("PHASE07_WORK"):
    WORK = Path(os.environ["PHASE07_WORK"])
else:
    found = [(p, _du(p)[1]) for p in CANDIDATES if Path(p).is_dir()]
    WORK = Path(found[0][0]) if found else Path.home() / "phase07"
    if found:
        print("candidates found:", ", ".join(f"{p} ({f:.0f} GB free)" for p, f in found))
        print(f"   -> using the first one: {WORK}   (set PHASE07_WORK=/path to override)")
WORK.mkdir(parents=True, exist_ok=True)

# Must be set before anything imports huggingface_hub, and they are inherited by lerobot-train.
os.environ["HF_HOME"] = str(WORK / "hf")
os.environ["HF_LEROBOT_HOME"] = str(WORK / "hf" / "lerobot")
OUTPUT_ROOT = WORK / "outputs"
for p in (Path(os.environ["HF_LEROBOT_HOME"]), OUTPUT_ROOT):
    p.mkdir(parents=True, exist_ok=True)

print("work dir   :", WORK)
print("HF cache   :", os.environ["HF_HOME"])
print("outputs    :", OUTPUT_ROOT)

mounts = {}
for p in {"/", str(Path.home()), str(WORK)}:
    try:
        t, f = _du(p)
        mounts[str(Path(p).resolve())] = (t, f)
    except Exception:
        pass
print("\nfilesystems:")
for p, (t, f) in sorted(mounts.items()):
    print(f"   {p:28s} {t:7.0f} GB total  {f:7.0f} GB free")

WORK_FREE = _du(WORK)[1]
if WORK_FREE < 90:
    print(f"\n!! Only {WORK_FREE:.0f} GB free in the working directory. The full run needs ~85 GB")
    print("   (14.5 base model + 0.7 dataset + 6 x 11.6 checkpoints). If this host has a bigger")
    print("   data disk, set PHASE07_WORK to it and re-run this cell BEFORE installing anything.")

# pip installs land on the ROOT filesystem, not the work dir: torch+cu130 and lerobot's dependencies
# are roughly 10-15 GB, and a container's overlay is often only 30 GB.
root_free = _du("/")[1]
print(f"\nroot filesystem free: {root_free:5.0f} GB   (pip puts ~10-15 GB of torch and deps here)")
if root_free < 20:
    print("!! Tight for the install. If it fails with 'No space left on device', install into the")
    print(f"   data disk instead:  %pip install --target {WORK}/site-packages ...")

# DataLoader workers talk through /dev/shm. A 64 MB default inside a container is the classic
# "DataLoader worker (pid N) is killed by signal: Bus error" with num_workers > 0.
try:
    shm_total = _du("/dev/shm")[0]
    ok_shm = shm_total >= 2
    print(f"/dev/shm            : {shm_total:5.0f} GB   "
          f"({'fine for 8 dataloader workers' if ok_shm else 'TOO SMALL -> use --num_workers=0'})")
except Exception:
    pass

## 2 — Credentials

On a rented box there is no Colab secret store, so put the two keys here. They are exported into the
process environment, which every `lerobot-train` subprocess inherits, and written to the Hugging Face
token file. **Nothing is printed back** — the cell only confirms a masked prefix.

Set `PERSIST_TO_SHELL = True` to also append them to `~/.bashrc`, so a reconnected SSH session or a
`tmux` pane has them without re-running the notebook. That writes the keys to disk in plain text, which
is normal practice on a throwaway rented box but worth doing knowingly.

The shell equivalent, if you would rather export them in a terminal before starting Jupyter:

```bash
export HF_TOKEN=hf_xxxxxxxxxxxxxxxxxxxxxxxx
export WANDB_API_KEY=xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
```

In [ ]:
import os
from pathlib import Path

# ---------------------------------------------------------------------------------------------
HF_TOKEN_VALUE      = ""      # https://huggingface.co/settings/tokens   (needs repo.write)
WANDB_API_KEY_VALUE = ""      # https://wandb.ai/authorize
PERSIST_TO_SHELL    = False   # also append both to ~/.bashrc for future shells
# ---------------------------------------------------------------------------------------------

for name, val in (("HF_TOKEN", HF_TOKEN_VALUE), ("WANDB_API_KEY", WANDB_API_KEY_VALUE)):
    if val:
        os.environ[name] = val.strip()

def _mask(v):
    return f"{v[:6]}...{v[-4:]} ({len(v)} chars)" if v and len(v) > 12 else ("set" if v else "MISSING")

for name in ("HF_TOKEN", "WANDB_API_KEY"):
    print(f"{name:14s} {_mask(os.environ.get(name, ''))}")

missing = [n for n in ("HF_TOKEN", "WANDB_API_KEY") if not os.environ.get(n)]
if missing:
    print(f"\n!! {missing} not set. Fill them in above, or export them in the shell before starting")
    print("   Jupyter. Section 5 will fall back to an interactive login, which does NOT reach the")
    print("   lerobot-train subprocess - so training would authenticate as nobody.")

if PERSIST_TO_SHELL:
    rc = Path.home() / ".bashrc"
    old = rc.read_text().splitlines() if rc.exists() else []
    keep = [l for l in old if not l.startswith(("export HF_TOKEN=", "export WANDB_API_KEY="))]
    keep += [f"export {n}={os.environ[n]}" for n in ("HF_TOKEN", "WANDB_API_KEY") if os.environ.get(n)]
    rc.write_text("\n".join(keep) + "\n")
    print(f"\nappended to {rc} (plain text) - new shells will have them")

## 3 — Install

A bare image has neither torch nor lerobot. **Torch comes first and from the CUDA 13.0 index**, because
installing lerobot alone would pull a default-CUDA wheel.

| | | |
|---|---|---|
| `torch` | **2.11.0+cu130** | lerobot v0.6.1 requires `torch<2.12,>=2.7`, so 2.11 is the newest it accepts |
| `torchvision` | **0.26.0+cu130** | the matching release for torch 2.11; also satisfies `torchvision<0.27` |
| `torchaudio` | **2.11.0+cu130** | matching release; not used directly, installed for a consistent trio |
| `lerobot` | **0.6.1** `[smolvla,dataset,training]` | the version that recorded the dataset |

The cell then **verifies what actually landed** rather than assuming: that torch is still the cu130
build after pip resolved lerobot's dependencies, that CUDA sees the GPU, and that `torchcodec` — which
`[dataset]` pulls — imports against this torch. A torchcodec built for a different torch still *exists*
as a module, so lerobot's `find_spec` check would select it and then fail at decode time; if that is the
case the cell sets `VIDEO_BACKEND = "pyav"` and the training command passes it explicitly.

> There is **no `act` extra** in v0.6.1 (pip accepts an unknown extra silently, which is why the Phase 0.6
> notebook's `lerobot[act]` appeared to work). `dataset` + `training` is what ACT actually needs.

### If pip says `externally-managed-environment`

Debian and Ubuntu ship a marker file (PEP 668) that makes pip refuse to touch the system Python. The cell
below **detects this and adds `--break-system-packages`** automatically — and skips the flag when it finds
a venv or conda environment, so the same cell works everywhere.

That flag is the right trade on a **disposable, single-purpose GPU box**: the thing PEP 668 protects is an
OS whose own tooling depends on the system Python, which is not what this machine is for. If you would
rather keep it pristine, make a venv *before* starting Jupyter and register it as the kernel:

```bash
python3 -m venv /root/venv
/root/venv/bin/pip install -q ipykernel
/root/venv/bin/python -m ipykernel install --user --name phase07 --display-name "phase07"
```

then pick the **phase07** kernel and re-run from the top; the cell will see `IN_ENV` and drop the flag.

> **Put the install on `/`, not on the data disk** — the default, and worth not "fixing". Torch plus the
> `nvidia-*` CUDA runtime wheels plus lerobot's dependencies come to roughly **10–15 GB**. The root
> overlay has 30 GB and nothing else to do with it; the data disk needs **84.5 GB** for the base model
> and six checkpoints, so moving the install there would leave about half a gigabyte of margin. Resist
> `--target`.

In [ ]:
import os, subprocess, sys, sysconfig
from pathlib import Path

# PEP 668: Debian and Ubuntu mark the system Python "externally managed", and pip then refuses to
# install into it ("error: externally-managed-environment"). Detect the situation rather than
# hard-coding a flag, so the same cell works in a venv, under conda, and on a bare system Python.
IN_ENV = sys.prefix != sys.base_prefix or bool(os.environ.get("CONDA_PREFIX"))
MANAGED = (Path(sysconfig.get_path("stdlib")) / "EXTERNALLY-MANAGED").exists()
PIP_EXTRA = [] if IN_ENV else (["--break-system-packages"] if MANAGED else [])

print("interpreter :", sys.executable)
print("in venv/conda:", IN_ENV, "| PEP 668 marker:", MANAGED)
if PIP_EXTRA:
    print("-> adding --break-system-packages: there is no separate environment to install into, and")
    print("   this is a disposable single-purpose box. See the markdown above for the venv route.")


def pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *PIP_EXTRA, *args]
    print("\n$", " ".join(cmd), flush=True)
    if subprocess.run(cmd).returncode:
        raise SystemExit("pip failed - see the output above")


# Torch FIRST and from the CUDA 13.0 index: installing lerobot first would pull a default-CUDA wheel.
pip("--index-url", "https://download.pytorch.org/whl/cu130",
    "torch==2.11.0+cu130", "torchvision==0.26.0+cu130", "torchaudio==2.11.0+cu130")
pip("lerobot[smolvla,dataset,training]==0.6.1", "wandb")
print("\ninstall finished")

In [ ]:
import importlib.util, subprocess, sys

import torch, lerobot
print("lerobot", lerobot.__version__)
print("torch  ", torch.__version__, "| built for CUDA", torch.version.cuda,
      "| cuda available:", torch.cuda.is_available())

assert torch.version.cuda and torch.version.cuda.startswith("13"), (
    f"torch reports CUDA {torch.version.cuda}, not 13.x - pip replaced the cu130 wheel while resolving "
    "lerobot. Re-run the install cell above, then restart the kernel.")
assert torch.cuda.is_available(), "torch cannot see the GPU - check the driver with nvidia-smi"

# torchcodec ships compiled against a specific torch; a mismatch imports-fails rather than
# resolve-fails, and lerobot selects it on find_spec alone, so prove it actually loads.
VIDEO_BACKEND = None
if importlib.util.find_spec("torchcodec"):
    r = subprocess.run([sys.executable, "-c", "import torchcodec; print(torchcodec.__version__)"],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print("torchcodec", r.stdout.strip(), "- usable, lerobot will decode with it")
    else:
        VIDEO_BACKEND = "pyav"
        print("torchcodec is installed but fails to import against this torch:")
        print("   ", (r.stderr.strip().splitlines() or ["?"])[-1][:160])
        print("-> forcing --dataset.video_backend=pyav (slower to seek, but dataloading is not the")
        print("   bottleneck here, and lerobot would otherwise pick torchcodec and fail at decode)")
else:
    print("torchcodec not installed - lerobot falls back to pyav on its own")

> If `import lerobot` or `import torch` fails here, restart the kernel (**Runtime > Restart session**, or
> the kernel menu) and re-run from this cell. Nothing later imports lerobot into this kernel — every use
> goes through a subprocess.

## 4 — Runtime check

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("torch", torch.__version__, "| cuda", torch.version.cuda, "|", GPU)
assert GPU, "No GPU visible. On Colab: Runtime > Change runtime type > GPU (L4)."
if "T4" in GPU:
    print("\n!! T4 (16 GB): too small for a useful batch here. Use an L4.")
elif "L4" in GPU:
    print("\nL4 (24 GB): measured 2026-10-06 - the full batch-64 recipe fits in 16.5 GiB with no")
    print("            step-down, ~11.4 h for 20k steps. This is the right GPU for SmolVLA; leave")
    print("            the A100 for pi0.5 so the two can run in parallel.")
elif "A100" in GPU:
    print("\nA100: works, but batch 64 only needs 16.5 GiB, so this is really an L4 job. If pi0.5")
    print("      is waiting on an A100, run SmolVLA on an L4 instead and start both at once.")

## 5 — Authentication

**A Colab secret alone is not enough.** `huggingface_hub` resolves a token as OIDC → `HF_TOKEN` env var →
token file → Colab secret, and that last path calls `google.colab.userdata.get()`, which needs the
kernel's channel to the Colab frontend. `lerobot-train` is a **subprocess**, where that channel does not
exist — so with only a secret set, `whoami()` succeeds in the notebook while training authenticates as
nobody. These cells copy the token into the environment *and* the token file, then prove a subprocess can
use it.

The run also creates a new model repo. A fine-grained token can be granted `repo.write` either globally
or **scoped to your own user** — the latter is the least-privilege setup and shows up nowhere under
`global`, so the cell checks both and only *warns* if it finds neither. Part B settles it for real by
pushing a checkpoint and verifying it landed.

In [ ]:
import os, sys, subprocess
from huggingface_hub import get_token, login, whoami

tok = os.environ.get("HF_TOKEN") or get_token()
if not tok:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
    except Exception:
        tok = None
if not tok:
    from huggingface_hub import notebook_login
    notebook_login(); tok = get_token()
assert tok, "no HF token - add a Colab secret named HF_TOKEN, or run notebook_login()"

login(token=tok, add_to_git_credential=False)     # writes ~/.cache/huggingface/token
os.environ["HF_TOKEN"] = tok                      # set AFTER login, so it does not warn that the
                                                  # env var overrides what was just configured
                                                  # (same token either way); subprocesses inherit it

r = subprocess.run([sys.executable, "-c",
                    "from huggingface_hub import whoami; print(whoami()['name'])"],
                   capture_output=True, text=True)
assert r.returncode == 0, "a subprocess still cannot authenticate:\n" + r.stderr[-800:]
me = whoami()
print("HF ok - notebook:", me["name"], "| subprocess:", r.stdout.strip())


def hf_write_grant(me):
    """Report how (or whether) this token is allowed to create repos in the user's namespace.

    A fine-grained token carries permissions either globally or SCOPED TO AN ENTITY. The
    least-privilege setup - and the one the Hub's UI steers you to - is `repo.write` scoped to your
    own user, which does not appear under `global` at all. Checking only `global` therefore rejects
    a perfectly good token, which is exactly what an earlier version of this cell did.
    """
    auth = me.get("auth", {}).get("accessToken", {})
    if auth.get("role") != "fineGrained":
        return True, f"{auth.get('role', 'write')} token"
    fg = auth.get("fineGrained") or {}
    if "repo.write" in (fg.get("global") or []):
        return True, "global repo.write"
    for s in fg.get("scoped") or []:
        ent = s.get("entity") or {}
        if (ent.get("type") in ("user", "org") and ent.get("name") == me["name"]
                and "repo.write" in (s.get("permissions") or [])):
            return True, f"repo.write scoped to {ent['type']} '{ent['name']}'"
    return False, f"no repo.write found - global={fg.get('global')}"


ok, how = hf_write_grant(me)
print("token write access:", how)
if not ok:
    # Deliberately a warning, not an assertion. The Hub's permission model has more shapes than it
    # is wise to enumerate here, and Part B proves the real thing 15 minutes from now by pushing a
    # checkpoint and checking it landed. A hard gate here can only produce false blocks.
    print("\n!! Could not confirm this token can create the model repo this run pushes to.")
    print("   If Part B's 'checkpoint uploaded to the Hub' check fails, that is the reason - give")
    print("   the token repo.write (globally, or scoped to your user) at")
    print("   https://huggingface.co/settings/tokens")

In [ ]:
import os, sys, subprocess, wandb

key = os.environ.get("WANDB_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
    except Exception:
        key = None
if key:
    os.environ["WANDB_API_KEY"] = key; wandb.login(key=key)
else:
    wandb.login()                                  # key from https://wandb.ai/authorize
    os.environ["WANDB_API_KEY"] = wandb.api.api_key

r = subprocess.run([sys.executable, "-c", "import wandb; print(bool(wandb.api.api_key))"],
                   capture_output=True, text=True)
assert r.stdout.strip() == "True", "a subprocess cannot see the wandb key:\n" + r.stderr[-500:]
print("wandb ok in notebook and subprocess")

## 6 — Configuration


| Knob | Value | Why |
|---|---|---|
| Base | `lerobot/smolvla_base` | The published checkpoint |
| Batch / steps | 64 / 20,000 | The published recipe, which **fits an L4 unmodified** (16.5 of 24 GiB). The smoke test still steps down and rescales steps if a smaller GPU OOMs |
| Adaptation | inherited: vision encoder frozen, action expert only | The base config's own setting - **99.9 M of ~450 M** parameters train, which is why batch 64 fits a 24 GB card. Printed in Part A so it is on the record |
| LR / schedule | 1e-4, cosine with 1k warmup | Policy preset. `scheduler_decay_steps` is pinned to the real step count so the cosine always completes |
| Images | 512×512 padded, both cameras | Base config |

In [ ]:
import collections, json, os, re, shutil, subprocess, sys, threading, time
from pathlib import Path
HF_USER = "HALDijkstraaa"
DATASET = f"{HF_USER}/so101_tray_cylinder_varied_20261013_082911_trimmed"
PROJECT = "phase07-policy-transfer"
SEED    = 1000

# Held out for validation loss. ROUND 2 NOTE: D1 randomises the cylinder position continuously, so
# the 9-block structure that forced a hand-picked holdout in round 1 is gone - the verifier measures
# 1x block separation where D0 had 27x, and lerobot's default "last N" split would now be perfectly
# fair. The same indices are kept anyway, so that the split MECHANISM is byte-identical to round 1
# and the only thing that differs between the two runs is the data itself. They are arbitrary here,
# which is the point: any 5 episodes of D1 are an equally fair holdout.
HOLDOUT  = [9, 19, 29, 39, 49]
EPISODES = [e for e in range(50) if e not in HOLDOUT] + HOLDOUT

EVAL_SPLIT, LOG_FREQ = 0.1, 200

# Keep every checkpoint on local disk. They are each the whole model plus optimizer state, so this
# is the setting that decides whether the run fits the disk - Part B projects the total before you
# start, and a watchdog re-checks it during the run and shouts before it runs out. Flip to False to
# retain only the newest KEEP_CKPTS instead; nothing is lost either way because every checkpoint is
# also pushed to the Hub.
KEEP_ALL_CHECKPOINTS = True
KEEP_CKPTS = 2

# ---- SmolVLA-specific ----------------------------------------------------------------------------
JOB   = "smolvla_tray_cylinder_s1000_d1"
# The published recipe. These two never change; BATCH/STEPS below are what the run actually uses, and
# Part B derives them from these so re-running it is idempotent rather than compounding.
BASE_BATCH, BASE_STEPS = 64, 30000
BATCH, STEPS = BASE_BATCH, BASE_STEPS
SAVE_FREQ = 5000      # how often a checkpoint is written AND pushed
EVAL_STEPS = 5000     # held-out loss cadence; training is unaffected either way.
# SAVE_FREQ must be a multiple of EVAL_STEPS, so that every checkpoint has an eval at its own
# step: lerobot tests `step % eval_steps == 0` and `step % save_freq == 0` independently, and
# runs the eval first, so a mismatched pair leaves some checkpoints with no loss beside them.
assert SAVE_FREQ % EVAL_STEPS == 0, f"{SAVE_FREQ} is not a multiple of {EVAL_STEPS}"
BASE_MODEL = "lerobot/smolvla_base"

# The pretrained checkpoint declares its own camera names; the batch is renamed to match.
RENAME = {
    "observation.images.top": "observation.images.camera1",
    "observation.images.wrist": "observation.images.camera2"
}

NUM_WORKERS = min(8, os.cpu_count() or 2)


def train_cmd(job=None, steps=None, batch=None, extra=()):
    """Build the lerobot-train argv.

    Every default is resolved from the globals at CALL time, not bound at definition time, so a
    batch step-down in Part B reaches a later bare train_cmd() call. Checkpoints go to the Hub every
    SAVE_FREQ steps, so a disconnect costs at most that many.
    """
    job = JOB if job is None else job
    steps = STEPS if steps is None else steps
    batch = BATCH if batch is None else batch
    return ["lerobot-train",
        f"--dataset.repo_id={DATASET}", f"--dataset.episodes={EPISODES}",
        f"--dataset.eval_split={EVAL_SPLIT}", f"--eval_steps={EVAL_STEPS}",
        *( [f"--dataset.video_backend={globals()['VIDEO_BACKEND']}"]
           if globals().get("VIDEO_BACKEND") else [] ),
        f"--policy.path={BASE_MODEL}",
        f"--rename_map={json.dumps(RENAME)}",

        "--policy.device=cuda",
        "--policy.push_to_hub=true", f"--policy.repo_id={HF_USER}/{job}", "--policy.private=true",
        "--save_checkpoint_to_hub=true",
        # Pin the cosine decay to the real step count so the schedule always completes,
        # whatever batch the smoke test settles on (the preset default is 30k).
        f"--policy.scheduler_decay_steps={steps}",
        f"--batch_size={batch}", f"--steps={steps}", f"--num_workers={NUM_WORKERS}",
        f"--seed={SEED}", f"--save_freq={SAVE_FREQ}", f"--log_freq={LOG_FREQ}",
        f"--output_dir={OUTPUT_ROOT}/{job}", f"--job_name={job}",
        "--wandb.enable=true", f"--wandb.project={PROJECT}", *extra]


KEEP = ("step:", "eval_loss", "Train/eval split", "num_learnable_params", "input_features",
        "Traceback", "Error", "error:", "Track this run", "End of training", "out of memory")


def gpu_watch(stop, peak):
    while not stop.is_set():
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.strip().splitlines()
        if out:
            peak[0] = max(peak[0], int(out[0]))
        time.sleep(5)


def disk_free_gb(path=None):
    # Falls through so a bad path can never raise inside the watchdog loop, where an exception
    # would silence the very warning this exists to produce.
    for p in ([str(path)] if path else []) + [str(WORK), "/", "."]:
        try:
            s = shutil.disk_usage(p)
            return s.free / 1e9, s.used / 1e9
        except Exception:
            continue
    return float("nan"), float("nan")


def dir_size_gb(path):
    try:
        return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e9
    except Exception:
        return 0.0


def watch_disk(stop, out_dir, total_steps, save_freq, period=120):
    """Report disk as the run proceeds and shout BEFORE it runs out.

    Every checkpoint is kept (KEEP_ALL_CHECKPOINTS), so the footprint grows all run: for pi0.5 that
    is ~11.5 GB every save_freq steps. Rather than discover the limit at hour 14, this measures the
    actual size of the checkpoints written so far, extrapolates to the number still to come, and
    compares that against free space.
    """
    d = Path(out_dir) / "checkpoints"
    warned = False
    while not stop.is_set():
        try:
            free, used = disk_free_gb(out_dir)
            done = sorted(int(p.name) for p in d.iterdir()
                          if d.is_dir() and p.is_dir() and not p.is_symlink() and p.name.isdigit()) \
                   if d.is_dir() else []
            if done:
                each = dir_size_gb(d) / len(done)
                remaining = max(0, (total_steps // save_freq) - len(done))
                need = each * remaining
                msg = (f"    [disk] {len(done)} checkpoint(s) written, {each:.1f} GB each | "
                       f"{remaining} to go needs ~{need:.0f} GB | {free:.0f} GB free")
                if need > free * 0.9 and not warned:
                    warned = True
                    print("\n" + "!" * 78, flush=True)
                    print(f"  DISK WARNING: the remaining {remaining} checkpoint(s) need "
                          f"~{need:.0f} GB but only {free:.0f} GB is free.", flush=True)
                    print(f"  This run will stop partway unless you act. Options, cheapest first:", flush=True)
                    print(f"    - grow the data disk (this host charges per GB-day);", flush=True)
                    print(f"    - delete older local checkpoints by hand - they are already on the", flush=True)
                    print(f"      Hub, so nothing is lost:", flush=True)
                    print(f"        import shutil; shutil.rmtree('{d}/<step>')", flush=True)
                    print(f"    - or set KEEP_ALL_CHECKPOINTS = False and restart the cell.", flush=True)
                    print("!" * 78 + "\n", flush=True)
                else:
                    print(msg, flush=True)
        except Exception as e:
            print(f"    [disk] watchdog pass failed ({type(e).__name__}); will retry", flush=True)
        stop.wait(period)


def reap_checkpoints(stop, out_dir, keep=KEEP_CKPTS):
    """Delete superseded LOCAL checkpoints. Only runs when KEEP_ALL_CHECKPOINTS is False.

    Each checkpoint is the full model plus optimizer state - for pi0.5 ~11.5 GB - and lerobot never
    prunes them. They are redundant locally the moment save_checkpoint_to_hub has pushed them, which
    is what makes deleting them safe. Keeps the newest `keep` numbered directories and never touches
    `last`, the symlink resume follows.
    """
    d = Path(out_dir) / "checkpoints"
    while not stop.is_set():
        try:
            if d.is_dir():
                steps = sorted((p for p in d.iterdir() if p.is_dir() and not p.is_symlink()
                                and p.name.isdigit()), key=lambda p: int(p.name))
                dropped = []
                for p in (steps[:-keep] if len(steps) > keep else []):
                    shutil.rmtree(p, ignore_errors=True)   # per-item, so one failure is not fatal
                    dropped.append(p.name)
                if dropped:
                    print(f"    [disk] removed local checkpoint(s) {', '.join(dropped)} "
                          f"(already on the Hub); {disk_free_gb()[0]:.0f} GB free", flush=True)
        except Exception as e:
            print(f"    [disk] reaper pass failed ({type(e).__name__}); will retry", flush=True)
        stop.wait(60)


def run(cmd, quiet=True):
    """Stream a run, dropping tqdm spam, and record peak GPU memory alongside throughput.

    Markers are extracted AS EACH LINE STREAMS BY, not by searching a buffer afterwards. A real run
    emits far more output than it is safe to hold in memory, and everything the smoke test checks -
    the train/eval split, the wandb URL, the parameter count - is printed in the first few seconds,
    so a bounded tail buffer would have evicted all of it long before the run ended.
    """
    t0 = time.time()
    head, tail = [], collections.deque(maxlen=120)   # head for context, tail for failure diagnosis
    peak, stop = [0], threading.Event()
    seen = {"split": None, "wandb": None, "params": None}
    rates, losses, eval_losses = [], [], []
    threading.Thread(target=gpu_watch, args=(stop, peak), daemon=True).start()
    out_dir = next((c.split("=", 1)[1] for c in cmd if c.startswith("--output_dir=")), None)
    n_steps = next((int(c.split("=", 1)[1]) for c in cmd if c.startswith("--steps=")), STEPS)
    if out_dir:
        if KEEP_ALL_CHECKPOINTS:
            threading.Thread(target=watch_disk, args=(stop, out_dir, n_steps, SAVE_FREQ),
                             daemon=True).start()
        else:
            threading.Thread(target=reap_checkpoints, args=(stop, out_dir), daemon=True).start()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    oom = False
    for line in p.stdout:
        if len(head) < 400:
            head.append(line)
        tail.append(line)
        if "out of memory" in line.lower():
            oom = True
        m = re.search(r"Train/eval split: ([0-9]+) train, ([0-9]+) eval", line)
        if m:
            seen["split"] = (int(m.group(1)), int(m.group(2)))
        m = re.search(r"(https://wandb\.ai/[^\s]+)", line)
        if m and not seen["wandb"]:
            seen["wandb"] = m.group(1)
        m = re.search(r"num_learnable_params=([0-9]+)", line)
        if m:
            seen["params"] = int(m.group(1))
        # tqdm FLIPS THE UNIT once an iteration takes over a second: ACT logs "4.47step/s" but
        # pi0.5 logs "1.95s/step". Parse both and normalise to steps/s, or the slow policies
        # report no throughput at all and the readiness verdict fails on a healthy run.
        rates += [float(x) for x in re.findall(r"([0-9.]+) ?step/s", line)]
        rates += [1.0 / float(x) for x in re.findall(r"([0-9.]+) ?s/step", line) if float(x) > 0]
        # " loss:" with the leading space on purpose: it must not also match l1_loss: or kld_loss:
        losses += [float(x) for x in re.findall(r" loss:([0-9.]+)", line)]
        eval_losses += [float(x) for x in re.findall(r"eval_loss=([0-9.]+)", line)]
        if (not quiet) or any(k in line for k in KEEP) or "wandb.ai" in line:
            sys.stdout.write(line); sys.stdout.flush()
    p.wait(); stop.set()
    dt = (time.time() - t0) / 3600
    free, used = disk_free_gb()
    print(f"[exit {p.returncode} after {dt:.2f} h | peak GPU {peak[0]/1024:.1f} GiB "
          f"| disk {used:.0f} GB used, {free:.0f} GB free]")
    if p.returncode != 0:
        print("--- last lines ---"); print("".join(tail)[-2500:])
    return {"code": p.returncode, "hours": dt, "peak_gib": peak[0] / 1024, "oom": oom,
            "rates": rates, "losses": losses, "eval_losses": eval_losses, **seen,
            "head": "".join(head), "tail": "".join(tail)}


print("holdout      :", HOLDOUT)
print("episode order:", EPISODES[:6], "...", EPISODES[-6:])
print("workers      :", NUM_WORKERS)

## 7 — Preflight

Three things that each cost minutes now and hours later: the dataset exists and carries its
codebase-version git tag; the holdout reorder is well formed; and the camera keys line up with what the
policy expects. None of this needs a GPU.

In [ ]:
import json
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
checks = []


ok_ds = api.repo_exists(DATASET, repo_type="dataset")
checks.append(("trimmed dataset exists", ok_ds))
assert ok_ds, f"{DATASET} not found. Run colab_phase07_prep_trim.ipynb (Stage 2) first."

info = json.loads(open(hf_hub_download(DATASET, "meta/info.json", repo_type="dataset")).read())
tags = [t.name for t in api.list_repo_refs(DATASET, repo_type="dataset").tags]
# lerobot resolves a dataset at the revision named by its codebase_version; without the tag, training
# dies several minutes in with "Your dataset must be tagged with a codebase version".
checks.append((f"codebase-version tag {info['codebase_version']}", info["codebase_version"] in tags))
checks.append(("50 episodes", info["total_episodes"] == 50))
checks.append(("idle-trimmed (fewer frames than the source's 35,141)", info["total_frames"] < 35141))
print(f"dataset: {info['total_episodes']} episodes, {info['total_frames']} frames, "
      f"{info['fps']} fps, tags={tags}")

checks.append(("holdout reorder is a permutation of 0-49", sorted(EPISODES) == list(range(50))))
checks.append(("holdout is the tail of the episode list", EPISODES[-len(HOLDOUT):] == HOLDOUT))

ds_cams = {k for k in info["features"] if k.startswith("observation.images.")}
# The decisive check for a pretrained policy: after renaming, the batch's camera keys must be a subset
# of what the checkpoint declares. lerobot accepts a subset in either direction, and anything the policy
# declares but the batch does not supply is padded internally.
pol_cfg = json.loads(open(hf_hub_download(BASE_MODEL, "config.json")).read())
pol_cams = {k for k, v in pol_cfg["input_features"].items() if v["type"] == "VISUAL"}
renamed = {RENAME.get(k, k) for k in ds_cams}
print("\npolicy declares :", sorted(pol_cams))
print("dataset supplies:", sorted(ds_cams))
print("after rename    :", sorted(renamed))
print("left unfed      :", sorted(pol_cams - renamed), "(padded internally with a zero mask)")
checks += [
    ("rename map keys exist in the dataset", set(RENAME) <= ds_cams),
    ("renamed keys are a subset of the policy's", renamed <= pol_cams),
]
for k, v in sorted(pol_cfg.items()):
    if k in ("freeze_vision_encoder", "train_expert_only", "optimizer_lr", "chunk_size",
             "n_action_steps", "dtype", "gradient_checkpointing", "resize_imgs_with_padding",
             "image_resolution"):
        print(f"  base config  {k:28s} = {v}")

print()
for n, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {n}")
assert all(ok for _, ok in checks), "preflight failed - fix before running Part B"
print("\npreflight OK")

---
# Part B — Smoke test

One cell, ~15 minutes. Runs **300 real training steps** — the balanced split, periodic validation, wandb,
and a checkpoint pushed to the Hub — then checks every result and prints a readiness verdict plus the
projected runtime for the full run on *your* GPU.

If it hits CUDA OOM it **halves the batch and retries**, then scales the step count to hold
samples-seen fixed, so the published recipe's sample budget survives a smaller GPU. Whatever it settles
on is what Part C uses.

It fails loudly rather than warning, so Part C cannot start on a broken setup.

In [ ]:
from huggingface_hub import HfApi

SMOKE_REPO, N = f"{HF_USER}/smolvla_smoke_delete_me", 300
batch, res = BATCH, None

while True:
    print(f"\n{'=' * 70}\n  smoke test at batch {batch}\n{'=' * 70}")
    shutil.rmtree(OUTPUT_ROOT / "smoke", ignore_errors=True)
    cmd = [c for c in train_cmd(job="smoke", steps=N, batch=batch)
           if not c.startswith(("--policy.repo_id", "--save_freq", "--eval_steps"))]
    cmd += [f"--policy.repo_id={SMOKE_REPO}", f"--eval_steps={N // 2}", f"--save_freq={N}"]
    res = run(cmd)

    if res["oom"] and batch > 1:
        print(f"\n!! OOM at batch {batch}. Halving and retrying.")
        print("   Steps will be scaled to keep samples-seen constant, so the recipe's sample budget")
        print("   is preserved even though the batch is smaller.")
        batch //= 2
        continue
    break

# Every check reports what was actually OBSERVED, so a failure is diagnosable rather than a bare
# FAIL. The values come from run(), which extracts them while the output streams past - searching a
# buffer afterwards is what made the first version report false failures, because the lines it
# looked for scroll away in the first seconds of a run.
split = res["split"]
checks = [
    ("training completed",
     res["code"] == 0),
    (f"balanced split 45 train / 5 eval   (saw {split})",
     split == (45, 5)),
    (f"validation ran   ({len(res['eval_losses'])} eval_loss points: {res['eval_losses']})",
     len(res["eval_losses"]) > 0),
    (f"wandb connected   ({res['wandb'] or 'NO run URL seen'})",
     bool(res["wandb"])),
]
try:
    files = HfApi().list_repo_files(SMOKE_REPO, repo_type="model")
    checks.append(("checkpoint uploaded to the Hub", any("checkpoints/" in f for f in files)))
except Exception as e:
    checks.append((f"checkpoint uploaded to the Hub ({type(e).__name__})", False))

losses, ev = res["losses"], res["eval_losses"]
if len(losses) >= 2:
    checks.append((f"train loss fell ({losses[0]:.3f} -> {losses[-1]:.3f})", losses[-1] < losses[0]))
if len(ev) >= 2:
    checks.append((f"eval loss fell ({ev[0]:.4f} -> {ev[-1]:.4f})", ev[-1] < ev[0]))

print()
for n_, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {n_}")

BATCH = batch
# Derived from the published recipe, not from the current STEPS, so re-running this cell after a
# step-down does not scale the step count a second time.
STEPS = int(round(BASE_STEPS * BASE_BATCH / BATCH))
rates = res["rates"]
rate = sorted(rates)[len(rates) // 2] if rates else None

print(f"\n{'=' * 70}")
print(f"  GPU            : {GPU}")
print(f"  peak memory    : {res['peak_gib']:.1f} GiB")
if rate:
    hours = STEPS / rate / 3600
    print(f"  throughput     : {rate:.2f} step/s  ({rate * BATCH:.0f} samples/s)")
    print(f"  batch          : {BATCH}"
          + ("" if BATCH == BASE_BATCH else f"   (stepped down from {BASE_BATCH})"))
    print(f"  steps          : {STEPS:,}   ({STEPS * BATCH / 1000:.0f}k samples)")
    print(f"  PROJECTED RUN  : {hours:.1f} h  (+ validation and checkpoint pushes, roughly 5%)")
    if hours > 12:
        print("\n  !! Over 12 h. Colab will likely disconnect at least once - Part C resumes from")
        print(f"     the last Hub checkpoint, losing at most {SAVE_FREQ:,} steps "
              f"({SAVE_FREQ / rate / 3600:.1f} h), so that is survivable.")
        print("     To shorten it instead, set STEPS by hand (the cosine schedule follows it) or")
        print("     try --policy.gradient_checkpointing=false if peak memory leaves room.")

    # Disk is the other thing that ends a long VLA run. Each checkpoint is the whole model plus
    # optimizer state, and the smoke test has just written exactly one, so measure rather than guess.
    ck = OUTPUT_ROOT / "smoke" / "checkpoints"
    sz = 0
    if ck.is_dir():
        sz = sum(f.stat().st_size for f in ck.rglob("*") if f.is_file()) / 1e9
    free, used = disk_free_gb()
    n_ck = max(1, STEPS // SAVE_FREQ)
    print(f"\n  disk now       : {used:.0f} GB used, {free:.0f} GB free")
    if sz:
        # Everything already on disk stays there (KEEP_ALL_CHECKPOINTS), so what matters is whether
        # the checkpoints still to be written fit in what is free RIGHT NOW.
        need = sz * n_ck
        fits = need < free
        print(f"  per checkpoint : {sz:.1f} GB  x {n_ck} checkpoints over the run = {need:.0f} GB")
        print(f"  keeping all    : {'FITS' if fits else 'WILL NOT FIT'}  "
              f"({need:.0f} GB needed vs {free:.0f} GB free, "
              f"{free - need:+.0f} GB margin at the end)")
        if not fits:
            print(f"\n  !! The run would stop partway. Either grow the data disk by "
                  f"~{need - free + 10:.0f} GB, raise SAVE_FREQ to write fewer checkpoints, or set")
            print(f"     KEEP_ALL_CHECKPOINTS = False to retain only the newest {KEEP_CKPTS}.")
        elif free - need < 2 * sz:
            print(f"     Margin is under two checkpoints - the watchdog in Part C will warn you if")
            print(f"     it tightens. Deleting an early checkpoint by hand is always safe; they are")
            print(f"     all on the Hub.")
        print(f"  Hub repo ends  : ~{need:.0f} GB, since every checkpoint is pushed (Part D prunes).")
else:
    print("  throughput     : could not parse step/s from the log")

SMOKE_OK = all(ok for _, ok in checks) and rate is not None
print(f"\n  READY: {SMOKE_OK}")
print("=" * 70)
if SMOKE_OK:
    # Reclaim the smoke run's own checkpoint before Part C starts. For pi0.5 that is ~11.5 GB,
    # a meaningful slice of the margin when every checkpoint is being kept.
    _before = disk_free_gb()[0]
    shutil.rmtree(OUTPUT_ROOT / "smoke", ignore_errors=True)
    _after = disk_free_gb()[0]
    print(f"\nsmoke output cleaned up: {_after - _before:+.1f} GB reclaimed, {_after:.0f} GB free")
    print("\nPaste the block above, then run Part C.")
    print(f"Clean up when convenient:  HfApi().delete_repo('{SMOKE_REPO}', repo_type='model')")
else:
    raise RuntimeError("smoke test failed - see the FAIL lines above. Do not start Part C.")

---
# Part C — Run it

**One cell. Run it and walk away.**

**Restartable.** If Colab disconnects, re-run this same cell. It first reads the run's Hub repo: a final
`model.safetensors` means finished (skip), checkpoints without one mean part-done (resume from the last,
losing at most 5,000 steps), neither means start. So a disconnect never repeats completed
work, and you never have to work out where it got to.

If you reconnected and `SMOKE_OK` is unset, that is expected — re-running Part B costs 15 minutes and
proves nothing new, so this cell warns rather than blocking.

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi

_api = HfApi()

if not globals().get("SMOKE_OK"):
    print("!! Part B has not run in this kernel (SMOKE_OK is unset).")
    print("   Expected after a reconnect: this cell is restartable and re-checks the Hub itself.")
    print("   On a fresh setup, run Part B first - it must print READY: True.\n")


def run_state(job):
    """Where this run stands, read from its Hub repo: finished, part-done, or not started."""
    try:
        files = _api.list_repo_files(f"{HF_USER}/{job}", repo_type="model")
    except Exception:
        return "new", None
    if "model.safetensors" in files:            # the final push only happens at the very end
        return "done", None
    steps = sorted({f.split("/")[1] for f in files
                    if f.startswith("checkpoints/") and f.split("/")[1].isdigit()})
    return ("resume", steps[-1]) if steps else ("new", None)

t0 = time.time()
where, last = run_state(JOB)

if where == "done":
    print(f"already finished on the Hub - nothing to do: https://huggingface.co/{HF_USER}/{JOB}")
    res = {"code": 0, "hours": 0.0}
elif where == "resume":
    # Prefer the local checkpoint if this VM still has it; otherwise pull from the Hub. The step
    # counter, optimizer state, episode order, holdout and seed all come back from the checkpoint -
    # only --steps is read from the CLI.
    local = OUTPUT_ROOT / JOB / "checkpoints/last/pretrained_model/train_config.json"
    src = str(local) if local.exists() else f"{HF_USER}/{JOB}"
    print(f"resuming from step {last} ({'local' if local.exists() else 'Hub'})")
    cmd = ["lerobot-train", f"--config_path={src}", "--resume=true", f"--steps={STEPS}"]
    if not local.exists():
        cmd.append(f"--output_dir={OUTPUT_ROOT}/{JOB}_resumed")
    res = run(cmd)
else:
    print(f"starting fresh: {STEPS:,} steps at batch {BATCH}")
    res = run(train_cmd(steps=STEPS, batch=BATCH))

status = "ok" if res["code"] == 0 else f"FAILED (exit {res['code']})"
print(f"\n{'=' * 70}")
print(f"  {JOB}")
print(f"  status : {status}")
print(f"  time   : {res['hours']:.2f} h")
print(f"  model  : https://huggingface.co/{HF_USER}/{JOB}")
print(f"  wandb  : project {PROJECT}, run {JOB}")
print("=" * 70)
if res["code"] != 0:
    print("\nRe-run this cell to resume from the last Hub checkpoint, or see Part D.")
else:
    print("\nPaste this summary back. Then run the other two policies' notebooks if they are not")
    print("already going, and record the result in Details/phase07_training.md.")

---
# Part D — Recovery

Only if something in Part C failed.

**Resume by hand.** Part C does this automatically, but if you want to force it:

```python
run(["lerobot-train", f"--config_path={HF_USER}/{JOB}", "--resume=true", f"--steps={STEPS}",
     "--output_dir={OUTPUT_ROOT}/resumed"])
```

**Start over.** Delete the model repo first, or `run_state` will see the old checkpoints and resume:

```python
HfApi().delete_repo(f"{HF_USER}/{JOB}", repo_type="model")   # irreversible
```

**OOM partway through a long run.** The batch that passed the smoke test should hold, but if a later
phase OOMs, drop the batch and resume — the optimizer state carries over:

```python
res = run(["lerobot-train", f"--config_path={HF_USER}/{JOB}", "--resume=true",
           f"--steps={STEPS}", f"--batch_size={BATCH // 2}"])
```

**It trained but you are not sure it trained on the right thing.** Check the run log for
`Train/eval split: 45 train, 5 eval` and that the episode order put `[9, 19, 29, 39, 49]` last. Both are
printed near the top of Part C's output and are also in the wandb config.

**Disk filled up anyway.** Part C reaps superseded *local* checkpoints automatically, keeping the newest
`KEEP_CKPTS` (= 2). Set it to 1 for more headroom, or clear the base-model download between runs with
`!rm -rf ~/.cache/huggingface/hub/models--*`. The current numbers are printed by Part B.

**Pruning the Hub repo.** Every checkpoint is pushed, so the repo ends up far larger than the final
model. Once you have settled on a checkpoint, delete the intermediate ones — this is irreversible, so
check what the run needs first (the 40k-vs-60k convergence comparison wants more than just the last one):

```python
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files(f"{HF_USER}/{JOB}", repo_type="model")
steps = sorted({f.split("/")[1] for f in files if f.startswith("checkpoints/")})
print("checkpoints on the Hub:", steps)
# keep the last two, drop the rest:
for s in steps[:-2]:
    api.delete_folder(f"checkpoints/{s}", repo_id=f"{HF_USER}/{JOB}", repo_type="model")
```

**Unfreezing the VLM.** The base config trains the action expert only. To fine-tune more of the model
(much slower, much more memory):

```python
res = run(train_cmd(job=JOB + "_full", extra=("--policy.train_expert_only=false",
                                              "--policy.freeze_vision_encoder=false")))
```

This would break the adaptation-regime match with the π0.5 run, so it is a separate experiment, not a
substitute.

---

### When all three are done

Collect the three summaries into `Details/phase07_training.md` — held-fixed table, per-policy recipe,
measured runtime and peak memory, final `eval_loss`, and the pre-registered predictions for the robot
rollouts. The three runs share the wandb project `phase07-policy-transfer`, so the eval-loss curves
overlay directly — but note that **eval loss is an action-prediction L1 and correlates only weakly with
task success** ([phase06 §2](Details/phase06_camera_ablation_training.md)); it is a divergence sanity
check, not the answer. The answer is rollouts on the robot.